# 04 Active users: characteristics and preferences

Test users are taken on their first test day and grouped by what they did the following week: no return, a return without a click, one or two clicks, and three clicks or more. Outputs: `25_*`.

## 1. Setup and groups

In [1]:
# Environment setup: libraries and shared project code
import json
import numpy as np
import pandas as pd

import thesis.click_model as cm
from thesis.config import P, O, TAG, connect
from thesis.panel import load, first_origin_tiers
from thesis.config import SEED
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score

In [2]:
# One row per test user at the user's first test origin, with next week's group and the predicted score
df, FEATS, tr, va, te = load()
S = pd.read_parquet(O / f"19_scores{TAG}.parquet")
first = first_origin_tiers(S)
u = (first.merge(te, on=["userId", "origin_day"], how="left")
          .merge(pd.read_parquet(P / "user_demographics.parquet"), on="userId", how="left"))
u["group"] = np.select([u.y_visit_days == 0, u.y_clicks == 0, u.y_clicks <= 2], [1, 2, 3], 4)
GROUPS = {1: "no return", 2: "returned, no click", 3: "1-2 clicks", 4: "3+ clicks"}
u.group.map(GROUPS).value_counts().reindex(GROUPS.values()).to_frame("users")

,users
group,
no return,80368
"returned, no click",47575
1-2 clicks,6852
3+ clicks,5346


## 2. Behavior up to the first test day

In [3]:
# Time of day, swipe-down use, and video share, popularity and age of cards shown and clicked; topics of the last 5 clicks
con = connect()
cm.prepare(con, P, df[df.bucket >= 85][["userId", "bucket"]].drop_duplicates("userId"))
con.register("t0", u[["userId", "origin_day"]])
beh = con.execute("""SELECT i.userId,
    avg(((i.impressTime / 1000 + 8 * 3600) % 86400 / 3600 < 6)::INT) AS night,
    avg(((i.impressTime / 1000 + 8 * 3600) % 86400 / 3600 >= 6 AND (i.impressTime / 1000 + 8 * 3600) % 86400 / 3600 < 12)::INT) AS morning,
    avg(((i.impressTime / 1000 + 8 * 3600) % 86400 / 3600 >= 12 AND (i.impressTime / 1000 + 8 * 3600) % 86400 / 3600 < 18)::INT) AS afternoon,
    avg(((i.impressTime / 1000 + 8 * 3600) % 86400 / 3600 >= 18)::INT) AS evening,
    avg(has_detail::INT) AS swipe_down_share,
    avg(video) AS video_shown, avg(CASE WHEN y = 1 THEN video END) AS video_clicked,
    avg(ln(1 + card_impr)) AS popularity_shown, avg(CASE WHEN y = 1 THEN ln(1 + card_impr) END) AS popularity_clicked,
    avg(card_age) AS card_age_shown, avg(CASE WHEN y = 1 THEN card_age END) AS card_age_clicked,
    sum(y) AS clicks_to_date
  FROM imp i JOIN t0 ON t0.userId = i.userId AND i.dt <= t0.origin_day GROUP BY i.userId""").df()
var = con.execute("""SELECT userId, count(DISTINCT talkId) AS topics_last5, count(DISTINCT creatorId) AS creators_last5 FROM (
    SELECT i.userId, i.talkId, i.creatorId,
           row_number() OVER (PARTITION BY i.userId ORDER BY i.impressTime DESC, i.row_id DESC) AS r
    FROM imp i JOIN t0 ON t0.userId = i.userId AND i.dt <= t0.origin_day WHERE i.y = 1)
  WHERE r <= 5 GROUP BY userId HAVING count(*) = 5""").df()
# impressions of these users up to the origin day, for the position-adjusted click rates of section 5
imp_t = con.execute("""SELECT i.userId, i.y, i.slot, i.depth, i.video, ln(1 + i.card_impr) AS popularity,
                              i.card_age FROM imp i JOIN t0 ON t0.userId = i.userId AND i.dt <= t0.origin_day""").df()
con.close()

## 3. Profiles by group

In [4]:
# Profile of each group: who, usage, preferences and variety
u = u.merge(beh, on="userId", how="left").merge(var, on="userId", how="left")


def num(c):
    return pd.to_numeric(u[c], errors="coerce")


u["age_n"] = num("age")
u["tenure_m"] = num("registeredMonthCnt")
u["level_n"] = num("level")
u["follow_n"] = num("followCnt")
u["age_missing"] = u.age_n.isna().astype(float)
u["female"] = (u.gender == "female").astype(float)
u["male"] = (u.gender == "male").astype(float)
u["new_account"] = (u.tenure_m <= 1).astype(float)
u["ctr_7d"] = u.w7_clicks / u.w7_impr
u["returned"] = (u.y_visit_days > 0).astype(float)
u["next_ctr"] = u.y_clicks / u.y_impr.where(u.y_impr > 0)

cols = {"users": ("userId", "size"), "mean_score": ("expected_click_prob", "mean"),
        # check: what happened next week
        "next_week_return": ("returned", "mean"), "next_week_clicks": ("y_clicks", "mean"),
        "next_week_click_rate": ("next_ctr", "mean"),
        # who
        "age_known_mean": ("age_n", "mean"), "age_gender_missing": ("age_missing", "mean"),
        "female": ("female", "mean"), "male": ("male", "mean"),
        "tenure_months_median": ("tenure_m", "median"), "new_account": ("new_account", "mean"),
        "level_mean_end_of_period": ("level_n", "mean"), "follow_median_end_of_period": ("follow_n", "median"),
        # usage
        "visit_days_7d": ("w7_visit_days", "mean"), "sessions_7d": ("w7_sessions", "mean"),
        "cards_shown_7d": ("w7_impr", "mean"), "clicks_7d": ("w7_clicks", "mean"), "click_rate_7d": ("ctr_7d", "mean"),
        "days_seen_in_log": ("history_days", "mean"), "mean_feed_position": ("h_mean_pos", "median"),
        "swipe_down_share": ("swipe_down_share", "mean"),
        "night": ("night", "mean"), "morning": ("morning", "mean"), "afternoon": ("afternoon", "mean"),
        "evening": ("evening", "mean"),
        # preferences: cards shown vs clicked
        "video_shown": ("video_shown", "mean"), "video_clicked": ("video_clicked", "mean"),
        "popularity_shown": ("popularity_shown", "mean"), "popularity_clicked": ("popularity_clicked", "mean"),
        "card_age_shown_days": ("card_age_shown", "mean"), "card_age_clicked_days": ("card_age_clicked", "mean"),
        # variety
        "share_with_5plus_clicks": ("topics_last5", lambda s: s.notna().mean()),
        "topics_in_last5_clicks": ("topics_last5", "mean"), "creators_in_last5_clicks": ("creators_last5", "mean")}
prof = u.groupby("group").agg(**cols)
prof.round(4).to_csv(O / f"25_rq1_profiles{TAG}.csv")
top = u.province.value_counts().head(8).index
(pd.crosstab(u.group, u.province.where(u.province.isin(top), "other"), normalize="index").round(4)
 .to_csv(O / f"25_rq1_provinces{TAG}.csv"))
prof.T.round(3)

group,1,2,3,4
users,80368.000,47575.000,6852.000,5346.000
mean_score,0.014,0.016,0.026,0.050
next_week_return,0.000,1.000,1.000,1.000
next_week_clicks,0.000,0.000,1.279,12.939
next_week_click_rate,NaN,0.000,0.089,0.131
age_known_mean,21.479,21.380,20.824,20.235
age_gender_missing,0.350,0.363,0.379,0.404
female,0.281,0.279,0.241,0.246
male,0.356,0.347,0.368,0.341
tenure_months_median,23.000,27.000,27.000,26.000


In [5]:
# Province shares by group (the eight largest provinces, the rest pooled)
pd.read_csv(O / f"25_rq1_provinces{TAG}.csv", index_col="group")

,guang dong,he nan,jiang su,other,shan dong,shan xi,si chuan,xin jiang,zhe jiang
group,,,,,,,,,
1,0.1175,0.0582,0.0691,0.4982,0.0595,0.0497,0.0513,0.0372,0.0594
2,0.1174,0.0540,0.0686,0.4971,0.0547,0.0519,0.0494,0.0469,0.0601
3,0.1188,0.0591,0.0639,0.4780,0.0569,0.0563,0.0504,0.0667,0.0499
4,0.1062,0.0550,0.0595,0.4957,0.0572,0.0629,0.0464,0.0711,0.0460


## 4. Variety
At equal volume, do users whose last five clicks span more topics return more often?

In [6]:
# OLS with robust (HC1) standard errors, one row per user with at least 5 clicks
v = u[u.topics_last5.notna()].copy()
X = pd.DataFrame({"const": 1.0, "topics_last5": v.topics_last5, "log_clicks_to_date": np.log1p(v.clicks_to_date),
                  "log_cards_7d": np.log1p(v.w7_impr), "visit_days_7d": v.w7_visit_days, "days_seen": v.history_days,
                  "log_tenure": np.log1p(v.tenure_m.fillna(v.tenure_m.median()))})


def ols_hc1(y, X):
    Xv, yv = X.values.astype(float), y.values.astype(float)
    XtX = np.linalg.inv(Xv.T @ Xv)
    b = XtX @ Xv.T @ yv
    e = yv - Xv @ b
    V = XtX @ (Xv.T * e ** 2) @ Xv @ XtX * len(yv) / (len(yv) - Xv.shape[1])
    se = np.sqrt(np.diag(V))
    return {c: {"coef": round(float(b[i]), 5), "ci95": [round(float(b[i] - 1.96 * se[i]), 5),
                                                        round(float(b[i] + 1.96 * se[i]), 5)]}
            for i, c in enumerate(X.columns)}


X_creators = X.assign(topics_last5=v.creators_last5).rename(columns={"topics_last5": "creators_last5"})
res = {"test_users": len(u), "users_with_5plus_clicks": len(v),
       "variety_hypothesis_return": ols_hc1(v.returned, X)["topics_last5"],
       "variety_hypothesis_visit_days": ols_hc1(v.y_visit_days, X)["topics_last5"],
       "variety_hypothesis_creators_return": ols_hc1(v.returned, X_creators)["creators_last5"],
       "note": "coefficients per extra distinct topic (creator) among the last 5 clicks, holding volume fixed"}
(O / f"25_rq1{TAG}.json").write_text(json.dumps(res, indent=2))
pd.DataFrame({k: v for k, v in res.items() if isinstance(v, dict)}).T

,coef,ci95
variety_hypothesis_return,-0.00788,"[-0.02131, 0.00556]"
variety_hypothesis_visit_days,-0.05261,"[-0.10403, -0.00119]"
variety_hypothesis_creators_return,-0.0006,"[-0.02654, 0.02533]"


## 5. Size of the differences

Standardized differences between the most active group and users who did not return, and between users who clicked and those who did not, also among users observed for at least eight days.

In [7]:
# Standardized differences with bootstrap intervals
u["log_cards_7d"] = np.log1p(u.w7_impr)
u["log_tenure"] = np.log1p(u.tenure_m)
u["log_mean_position"] = np.log1p(u.h_mean_pos)
u["any_click"] = (u.y_clicks > 0).astype(int)
ATTR = [("who", "age_n", "Age (where known)"), ("who", "age_missing", "Age and gender missing"),
        ("who", "female", "Female"), ("who", "male", "Male"), ("who", "log_tenure", "Account age (log months)"),
        ("who", "new_account", "New account"),
        ("usage", "w7_visit_days", "Visit days, last 7"), ("usage", "w7_sessions", "Sessions, last 7 days"),
        ("usage", "log_cards_7d", "Cards shown, last 7 days (log)"), ("usage", "ctr_7d", "Click rate, last 7 days"),
        ("usage", "history_days", "Days since first seen"), ("usage", "log_mean_position", "Feed position (log)"),
        ("usage", "swipe_down_share", "Swipe-down share"), ("usage", "evening", "Share of cards seen in the evening"),
        ("usage", "night", "Share of cards seen at night"),
        ("content", "video_shown", "Video share of cards shown"), ("content", "popularity_shown", "Popularity of cards shown"),
        ("content", "card_age_shown", "Age of cards shown")]
rng = np.random.default_rng(SEED)


def standardised_differences(lo, hi, label, reps=500):
    rows = []
    ia = [rng.integers(0, len(lo), len(lo)) for _ in range(reps)]
    ib = [rng.integers(0, len(hi), len(hi)) for _ in range(reps)]
    for group, col, name in ATTR:
        x1, x2 = lo[col].to_numpy(float), hi[col].to_numpy(float)

        def smd(v1, v2):
            v1, v2 = v1[~np.isnan(v1)], v2[~np.isnan(v2)]
            return (v2.mean() - v1.mean()) / np.sqrt((v1.var(ddof=1) + v2.var(ddof=1)) / 2)
        boot = [smd(x1[i], x2[j]) for i, j in zip(ia, ib)]
        rows.append({"sample": label, "group": group, "attribute": col, "label": name,
                     "mean_low": np.nanmean(x1), "mean_high": np.nanmean(x2), "smd": smd(x1, x2),
                     "smd_low": np.quantile(boot, 0.025), "smd_high": np.quantile(boot, 0.975),
                     "users_low": int((~np.isnan(x1)).sum()), "users_high": int((~np.isnan(x2)).sum())})
    return rows


long = u[u.history_days >= 8]
diffs = pd.DataFrame(standardised_differences(u[u.group == 1], u[u.group == 4], "3plus_vs_no_return_all") +
                     standardised_differences(long[long.group == 1], long[long.group == 4], "3plus_vs_no_return_8days") +
                     standardised_differences(u[u.any_click == 0], u[u.any_click == 1], "click_vs_none_all") +
                     standardised_differences(long[long.any_click == 0], long[long.any_click == 1], "click_vs_none_8days"))
diffs.round(4).to_csv(O / f"25_rq1_differences{TAG}.csv", index=False)
print(f"{len(long):,} users observed for at least 8 days ({len(long) / len(u):.0%})")
diffs.pivot(index="label", columns="sample", values="smd").round(2).loc[[n for _, _, n in ATTR]]

67,975 users observed for at least 8 days (49%)


sample,3plus_vs_no_return_8days,3plus_vs_no_return_all,click_vs_none_8days,click_vs_none_all
label,,,,
Age (where known),-0.33,-0.34,-0.24,-0.24
Age and gender missing,0.12,0.11,0.08,0.07
Female,-0.08,-0.08,-0.08,-0.08
Male,-0.04,-0.03,0.00,0.01
Account age (log months),0.09,0.07,0.05,0.04
New account,-0.09,-0.06,-0.06,-0.02
"Visit days, last 7",1.20,1.09,0.65,0.68
"Sessions, last 7 days",0.90,0.83,0.56,0.56
"Cards shown, last 7 days (log)",1.53,1.46,0.92,0.93


## 6. Demographics against behavior

How well each ranks next week's clickers and returning users, by cross-validated AUC.

In [8]:
# Cross-validated AUC of logistic regressions on demographics, on recent behavior, and on both
def dummies(s, prefix):
    return pd.get_dummies(s.astype(str), prefix=prefix, drop_first=True).astype(float)


age_band = pd.cut(u.age_n, [0, 17, 21, 25, 30, 200], labels=["<18", "18-21", "22-25", "26-30", "31+"]).astype(str)
DEMO = pd.concat([dummies(age_band.where(u.age_n.notna(), "missing"), "age"),
                  dummies(u.gender.fillna("missing"), "gender"),
                  dummies(pd.cut(u.tenure_m, [-1, 1, 6, 12, 24, 48, 1000]).astype(str), "tenure"),
                  dummies(u.province.where(u.province.isin(top), "other"), "prov")], axis=1)
USE = pd.DataFrame({"visit_days": u.w7_visit_days, "sessions": np.log1p(u.w7_sessions), "cards": u.log_cards_7d,
                    "clicks": np.log1p(u.w7_clicks), "history": u.history_days, "position": u.log_mean_position.fillna(0)})


def cv_auc(X, y):
    X = X.to_numpy(float)
    X = (X - X.mean(0)) / (X.std(0) + 1e-9)
    p = np.zeros(len(y))
    for a, b in StratifiedKFold(5, shuffle=True, random_state=SEED).split(X, y):
        p[b] = LogisticRegression(max_iter=2000).fit(X[a], y[a]).predict_proba(X[b])[:, 1]
    return round(float(roc_auc_score(y, p)), 4)


explained = {t: {"demographics": cv_auc(DEMO, u[t].values), "behaviour_last_7_days": cv_auc(USE, u[t].values),
                 "both": cv_auc(pd.concat([DEMO, USE], axis=1), u[t].values)}
             for t in ("any_click", "returned")}
explained["users"] = len(u)
pd.DataFrame({k: v for k, v in explained.items() if isinstance(v, dict)})

,any_click,returned
demographics,0.5729,0.5512
behaviour_last_7_days,0.7657,0.7272
both,0.7715,0.7309


## 7. What users click, net of where cards were shown
Clicks on each kind of card divided by those expected from the user's own click rate and the card's slot and page depth.

In [9]:
# Relative click rate by kind of card, within group, net of the user's own click rate and the card's position
it = imp_t.merge(u[["userId", "group"]], on="userId")
it["cell"] = it.slot.astype(int) * 5 + it.depth.astype(int)
it["popularity_third"] = pd.qcut(it.popularity, 3, labels=False, duplicates="drop")
it["age_third"] = pd.qcut(it.card_age.astype(float), 3, labels=False, duplicates="drop")
KINDS = [("video", 1, "Video"), ("video", 0, "Not video"),
         ("popularity_third", 0, "Least popular third"), ("popularity_third", 2, "Most popular third"),
         ("age_third", 0, "Newest third"), ("age_third", 2, "Oldest third")]
rel = []
for grp_, g in it.groupby("group"):
    uc = g.userId.astype("category").cat.codes.to_numpy()
    y, cell = g.y.to_numpy(float), g.cell.to_numpy()
    nu = uc.max() + 1
    uf, pf = np.ones(nu), np.ones(30)
    for _ in range(20):                                       # alternating updates of the user and position factors
        pf = np.bincount(cell, y, 30) / np.maximum(np.bincount(cell, uf[uc], 30), 1e-12)
        uf = np.bincount(uc, y, nu) / np.maximum(np.bincount(uc, pf[cell], nu), 1e-12)
    x = uf[uc] * pf[cell]
    boot_w = [np.bincount(rng.integers(0, nu, nu), minlength=nu) for _ in range(200)]
    for col, level, name in KINDS:
        m_ = (g[col] == level).to_numpy()
        yk, xk = np.bincount(uc[m_], y[m_], nu), np.bincount(uc[m_], x[m_], nu)
        reps_ = [(w @ yk) / (w @ xk) for w in boot_w]
        rel.append({"group": grp_, "kind": col, "level": level, "label": name, "clicks": int(yk.sum()),
                    "relative_click_rate": yk.sum() / xk.sum(),
                    "low": np.quantile(reps_, 0.025), "high": np.quantile(reps_, 0.975)})
rel = pd.DataFrame(rel)
rel.round(4).to_csv(O / f"25_rq1_relative_click_rates{TAG}.csv", index=False)
rel.pivot(index="group", columns="label", values="relative_click_rate").round(2)

label,Least popular third,Most popular third,Newest third,Not video,Oldest third,Video
group,,,,,,
1,0.90,1.05,0.90,1.02,0.99,0.97
2,0.95,1.01,0.94,1.04,0.98,0.94
3,0.95,1.01,0.96,1.04,0.99,0.95
4,0.98,0.99,0.96,1.02,1.01,0.98


In [10]:
# Save
res.update({"auc_demographics_vs_behaviour": explained, "users_observed_8_days_or_more": int(len(long)),
            "group_sizes": {GROUPS[k]: int(v) for k, v in u.group.value_counts().sort_index().items()},
            "group_sizes_8_days": {GROUPS[k]: int(v) for k, v in long.group.value_counts().sort_index().items()}})
(O / f"25_rq1{TAG}.json").write_text(json.dumps(res, indent=2))

1107

## 8. Summary
Active users differ from the rest in how often they visit and how they use the feed, not in who they are.